In [1]:
import pandas as pd
import numpy as np
import requests
import json
from bs4 import BeautifulSoup
from unidecode import unidecode

# Data Scraping
In this notebook we will use the author names from the Wikipedia dataset to scrape the names and years of their published works from fantasticfiction.com. Fantastic Fiction only has data on novels, so we first extract a list of all the novelists, then modify their names to a format which matches Fantastic Fiction's query system. We first checked Fantastic Fiction's robots.txt and ensured that our usage was permitted. We also made considerations to avoid sending too many rapid requests to the website.

In [2]:
# Read Laureates data
df = pd.read_csv("data/laureates_clean.csv")
# Extract authors who were primarily novelists
novelists = df[df["Genre"]=="Novel"].reset_index(drop=True)

# Decode each author's name to remove accents and save as "Decoded Name" feature
decoded_names = []
for i in range(novelists.shape[0]):
    decoded_names.append(unidecode(novelists.iloc[i]["Name"]))
    
dec = pd.Series(decoded_names, name="Decoded Name")
novelists = pd.concat([novelists, dec], axis=1)

# Add exceptions to decoded names to match the source we are scraping data from
novelists["Decoded Name"] = novelists["Decoded Name"].replace(
    {"Vidiadhar Surajprasad Naipaul": "V S Naipaul",
     "John Maxwell Coetzee":"J M Coetzee",
     "Jean-Marie Gustave Le Clezio":"J M G Le Clezio",
     "Pearl Buck":"Pearl S Buck",
     }
)

# Make Decoded Name lowercase with dashes instead of spaces to match url format
novelists["Decoded Name"] = novelists["Decoded Name"].str.lower().str.replace(" ","-",regex=True)
# Obtain the first letter of each novelist's last name
novelists["Letter"] = novelists["Decoded Name"].str.extract(r'-([a-z])[a-z]*$')

# Add exceptions for last names to match the data source
novelists.loc[novelists["Decoded Name"] == 'yasunari-kawabata', "Letter"] = "y"
novelists.loc[novelists["Decoded Name"] == 'gabriel-garcia-marquez', "Letter"] = "g"
novelists.loc[novelists["Decoded Name"] == 'kenzaburo-oe', "Letter"] = "k"
novelists.loc[novelists["Decoded Name"] == 'gao-xingjian', "Letter"] = "g"


To obtain the works of each author, we will use BeautifulSoup to parse the HTML content of the author's page of Fantastic Fiction. We will extract the titles and publication years of their novels and store this information in a structured format for further analysis.

(WARNING: THIS CELL WILL TAKE AROUND 80 SECONDS TO RUN, SINCE time.sleep(1) IS USED TO DELAY requests CALLS).

In [3]:
import time

works = []
headers = {"User-Agent":"COP 4283 Educational Project (UCF)"}

# For each author
for i in range(novelists.shape[0]):
    # Sleep for 1 second to avoid spamming the webpage
    time.sleep(1)
    
    # Get HTML page for the author (if there is one)
    name = novelists.iloc[i]["Decoded Name"]
    print(name)
    letter = novelists.iloc[i]["Letter"]
    r = requests.get(f'https://www.fantasticfiction.com/{letter}/{name}/', headers=headers)
    if r.status_code == 404:
        works.append(pd.NA)
        continue
    
    # Parse the page with beautiful soup, then search the tags for "Novels" and "Series"
    data = json.dumps(r.text)
    soup = BeautifulSoup(data, "html.parser")
    divs = soup.find_all("div")
    series_mark = -1
    novel_mark = -1
    
    for j,div in enumerate(divs):
        if(div.getText().lower()=="novels"):
            novel_mark = j
        if(div.getText().lower()=="series"):
            series_mark = j
    
    titles = []
    year_list = []
    
    # Scrape the titles and publication years of each book from the Series tab
    if series_mark >= 0:
        series = divs[series_mark-1]
        ser = series.find_all("a")
        items = []
        for h in ser:
            if h.getText():
                items.append(h)
        for k in items:
            if not k.find("span",{"class":r'\"ibig\"'}):
                titles.append(k.getText())
        years = series.find_all("span",{"class":r'\"year\"'})
        for y in years:
            year_list.append(y.getText())
    
    # Scrape the titles and publication years of each book from the Novels tab
    if novel_mark >= 0:
        books = divs[novel_mark+1].find_all("a")
        for book in books:
            titles.append(book.getText())
        years = divs[novel_mark+1].find_all("span",{"class":r'\"year\"'})
        for year in years:
            year_list.append(year.getText())
    
    # Handle some exceptions due to quirks of the webpage
    if name == "j-m-g-le-clezio":
        titles.remove('Patrick Modiano')
    if name == "patrick-modiano":
        titles.remove('J M G Le Clézio')
    
    # Print the titles and years obtained for the author
    print(titles)
    print(len(titles))
    print(year_list)
    print(len(year_list))
    
    # Associate a DataFrame of titles and years to the corresponding author.
    works.append(pd.DataFrame({"Title":titles, "Year":year_list}))


henryk-sienkiewicz
['With Fire and Sword', 'The Deluge', 'Fire in the Steppe', 'In Vain', 'Pan Michael', 'Without Dogma', 'Quo Vadis?', 'So Runs The World', 'The Knights of the Cross', 'The Field of Glory', 'In Desert and Wilderness', 'Whirlpools', 'Through the Desert', 'The Teutonic Knights', 'Dust and Ashes or Demolished', 'In Monte Carlo']
16
[' (1884)', ' (1886)', ' (1888)', ' (1872)', ' (1888)', ' (1891)', ' (1896)', ' (1898)', ' (1900)', ' (1906)', ' (1911)', ' (1911)', ' (1912)', ' (1993)', ' (2002)', ' (2002)']
16
rudyard-kipling
['The Jungle Book', 'The Second Jungle Book', 'The Jungle Books', 'The Complete Jungle Book', "Puck of Pook's Hill", 'Rewards and Fairies', 'The Light That Failed', 'Naulahka', 'Captains Courageous', 'Under the Deodars', 'Kim', 'They', 'With the Night Mail', 'The Legs of Sister Ursula']
14
[' (1894)', ' (1895)', ' (2017)', ' (2018)', ' (1906)', ' (1910)', ' (1891)', ' (1892)', ' (1897)', ' (1899)', ' (1901)', ' (1905)', ' (1909)', ' (1927)']
14
selma-l

In [4]:
# Append the series of Works DataFrames to the novelists DataFrame
novelists["Works"] = pd.Series(works)

In [5]:
# For each novelist with data available, clean the Years column to remove whitespace/parentheses 
# and make it numeric
present_novelists = novelists[~novelists.isna()["Works"]].reset_index(drop=True)

for i in present_novelists.index:
    present_novelists.loc[i,"Works"].loc[:,"Year"] = present_novelists.loc[i,"Works"].loc[:,"Year"].str.strip().str.replace(r"[()]","",regex=True).astype(int)

In [6]:
# Print the cleaned "Works" column
for i in present_novelists.index:
    print(present_novelists.loc[i,"Works"])

                           Title  Year
0            With Fire and Sword  1884
1                     The Deluge  1886
2             Fire in the Steppe  1888
3                        In Vain  1872
4                    Pan Michael  1888
5                  Without Dogma  1891
6                     Quo Vadis?  1896
7              So Runs The World  1898
8       The Knights of the Cross  1900
9             The Field of Glory  1906
10      In Desert and Wilderness  1911
11                    Whirlpools  1911
12            Through the Desert  1912
13          The Teutonic Knights  1993
14  Dust and Ashes or Demolished  2002
15                In Monte Carlo  2002
                        Title  Year
0             The Jungle Book  1894
1      The Second Jungle Book  1895
2            The Jungle Books  2017
3    The Complete Jungle Book  2018
4         Puck of Pook's Hill  1906
5         Rewards and Fairies  1910
6       The Light That Failed  1891
7                    Naulahka  1892
8         Cap

Since the "Works" column contains DataFrames, we save the novelists DataFrame using the pickle format to preserve this structure.

In [ ]:
# Export the novelists DataFrame to a pickle object for later use.
present_novelists.to_pickle('data/novelists.pkl') 